In [77]:
# init
from importlib import reload
import os
from pathlib import Path
import pandas as pd
from IPython.display import clear_output
import boto3
import sys
import time

import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsOSM.overpass as too
import toolsPandas.helpers as tph
import toolsSync.main as tsm

def pckgs_reload():
    reload(tgm)
    reload(too)
    reload(tph)
    reload(tgl)
    reload(tsm)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
TESTS_DIR = DATA_DIR / 'tests results'
CLEANED_DIR = DATA_DIR / 'cleaned'
DEV_MODE = True

task = 'test_duplicates'
TEST_DUPLICATES_DIR = TESTS_DIR / 'osm duplicates test'
process_state = tgm.load(DATA_DIR / "process_state.json")
dups_test_state = tgm.load(DATA_DIR / "dups_test_state.json")

logger = tgl.initiate_logger('logger', TEST_DUPLICATES_DIR / 'dups_test.log')

### [dev]

In [ ]:
temp1 = too.is_child_inside_parent('6543282','6543265')
[[k,v['result']] for k,v in temp1.items()]

[INFO] :    > Getting admin_centre:
[INFO] :     * Found node (admin_centre)
[INFO] :    > Testing admin_centre node (id: 7934379969)
[INFO] :     * Finished testing (admin_centre): False
[INFO] :    > Getting label:
[INFO] :     * Missing node (label)
[INFO] :    > Getting place:
[INFO] :     * Found node (place)
[INFO] :    > Testing place node (id: 7934379969)
[INFO] :     * Finished testing (place): False
[INFO] :    > Getting geom_node:
[INFO] :     * Found node (geom_node)
[INFO] :    > Testing geom_node node (id: 4298631214)
[INFO] :     * Finished testing (geom_node): True
[INFO] :    > Getting centroid:
[INFO] :     * Found node (centroid)
[INFO] :    > Testing centroid (lat: 36.8944071, lon: 6.4872707)
[INFO] :     * Finished testing (centroid): True


[['admin_centre', False],
 ['label', None],
 ['place', False],
 ['geom_node', True],
 ['centroid', True]]

In [ ]:
tph.get_from_df(df_all,['id','tags.parent_id','tags.country_id'],('6543282', '6543265', '192756'))

,type,id,tags.admin_level,tags.parent_id,tags.name,tags.name:us,tags.ISO3166-1,tags.ISO3166-2,tags.is_in:country,tags.ref:nuts,tags.ref:nuts:2,tags.ref:nuts:3,tags.addr:country,tags.country_name,tags.country_id
2462,rel,6543282,8,6543265,Beni Zid ⴱⴻⵏⵉ ⵣⵉⴷ بنى زيد,<NA>,<NA>,<NA>,Algeria,<NA>,<NA>,<NA>,<NA>,Algeria,192756


### * select to test

In [1]:
countries_tested = [c for c, val in process_state.items() if (val[task]['status'] == 'ok')]
logger.info(f"countries tested: {len(countries_tested)}")
countries_to_test = [c for c, val in process_state.items() if 
    (val['clean']['status'] == 'ok') and (val[task]['status'] in ['pending', 'error'])
]
logger.info(f"countries to test: {len(countries_to_test)}")

[INFO] : countries tested: 31
[INFO] : countries to test: 1


### * initialize B2

In [2]:
session = boto3.session.Session()

s3 = session.client(
    service_name="s3",
    aws_access_key_id=os.environ["B2_KEY_ID"],
    aws_secret_access_key=os.environ["B2_APPLICATION_KEY"],
    endpoint_url=os.environ["B2_ENDPOINT"]
)

### * download required data

In [3]:
logger.info(f"* Downloading required data to test: {len(countries_to_test)} countries")
logger.info(f"  * Downloading data from B2 in directory: '{CLEANED_DIR.relative_to(ROOT)}'")

countries_downloaded = tsm.donwload_country_data_from_bucket(countries_to_test, os.environ["B2_BUCKET_NAME"], CLEANED_DIR.relative_to(ROOT), CLEANED_DIR, s3, logger)

logger.info(f'* Countries to test: {len(countries_to_test)}')
logger.info(f"* Countries to test with downloaded cleaned data from B2: {len(countries_downloaded)}")

[INFO] : * Downloading required data to test: 2 countries
[INFO] :   * Downloading data from B2 in directory: 'data\cleaned'
[INFO] :   * Countries to get data: 2
[INFO] :     * Found in B2: 2, Missing in B2: 0
[INFO] :   * Downloading data for countries: 2
[INFO] :     * Directory: 'data\cleaned' -> 'c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned'
[INFO] :     * (1/2) Country UnitedStates files found: 1
[INFO] :       * Skip download of existing file UnitedStates_cleaned.pkl
[INFO] :     * (2/2) Country Bolivia files found: 1
[INFO] :       * Skip download of existing file Bolivia_cleaned.pkl
[INFO] :   * Number of downloaded files: 2/2
[INFO] : * Countries to test: 2
[INFO] : * Countries to test with downloaded cleaned data from B2: 2


### * load data for countries to test

In [3]:
logger.info(f"* Load data from: {CLEANED_DIR.relative_to(ROOT)}")
countries_to_test_df = tgm.load_single_file_dirs(CLEANED_DIR, countries_to_test)
logger.info(f"  * Countries to test {len(countries_to_test)} ; countries with data loaded {len(countries_to_test_df)}")

[INFO] : * Load data from: data\cleaned
[INFO] :   * Countries to test 1 ; countries with data loaded 1


### * select relations with duplicates ids

In [ ]:
# dups_id is computed using all countries in cleaned data, so we need just to filter here
dups_id = tgm.load(TEST_DUPLICATES_DIR  / 'dups_id.pkl')
logger.info(f"Duplicates ids: {len(dups_id)}")

[INFO] : Duplicates ids: 2554


In [5]:
logger.info(f"Countries to test: {len(countries_to_test_df)}")
logger.info(f"Relations to test: {len([row['id'] for df in countries_to_test_df.values() for i, row in df.iterrows()])}")

[INFO] : Countries to test: 1
[INFO] : Relations to test: 32903


In [ ]:
dups_df = {}
countries_wihout_first_level = []
for country, df in countries_to_test_df.items():
    dups = df[df['id'].isin(dups_id)]
    if not dups.empty:
        dups_df[country] = dups
    else:
        countries_wihout_first_level.append(country)
        tsm.update_process_state(process_state, country, task, process_status='missing')
        tgm.dump(DATA_DIR / "process_state.json", process_state)

logger.info(f"countries with first level: {len(dups_df)} \n {list(dups_df.keys())}")
logger.info(f"relations duplicates to test: {len([row['id'] for df in dups_df.values() for i, row in df.iterrows()])}")
logger.info(f"countries without first level: {len(countries_wihout_first_level)} \n{countries_wihout_first_level}")

[INFO] : countries with first level: 1 
 ['UnitedStates']
[INFO] : countries without first level: 0 
[]


In [73]:
dups_df['UnitedStates'].query("id == '8309397'")

,type,id,tags.admin_level,tags.parent_id,tags.name,tags.name:en,tags.alt_name:en,tags.ISO3166-1,tags.ISO3166-2,tags.is_in:country,tags.ref:nuts,tags.ref:nuts:2,tags.ref:nuts:3,tags.addr:country,tags.country_name,tags.country_id
216,rel,8309397,6,391455,Inuvialuit Settlement Region,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,UnitedStates,148838
1210,rel,8309397,6,1116270,Inuvialuit Settlement Region,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,UnitedStates,148838


### * exclude processed relations

In [8]:
dups_test_state = tgm.load(DATA_DIR / 'dups_test_state.json')
logger.info(f"Countries dups state: {len(dups_test_state)}")

processed_id_triplets = [ids for res in dups_test_state.values() for ids in res['processed']]
logger.info(f"Processed id triplets: {len(processed_id_triplets)}")

[INFO] : Countries dups state: 214
[INFO] : Processed id triplets: 1865


In [9]:
logger.info(f"Countries to process: {len(dups_df)}")
logger.info(f"Relations to process: {len([row['id'] for df in dups_df.values() for i, row in df.iterrows()])}")

[INFO] : Countries to process: 1
[INFO] : Relations to process: 4160


In [ ]:
dups_pending_process_df = {}
for country, df in dups_df.items():
    processed = dups_test_state[country]['processed'] if dups_test_state.get(country) else set()
    failed = dups_test_state[country]['failed'] if dups_test_state.get(country) else set()

    in_processed = df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(processed)
    in_failed = df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(failed)
    filtered_df = df[~in_processed | in_failed]
    if not filtered_df.empty:
        dups_pending_process_df[country] = filtered_df

In [11]:
print(f"Countries to process: {len(dups_pending_process_df)}")
print(f"Relations to process filtered: {len([row['id'] for df in dups_pending_process_df.values() for i, row in df.iterrows()])}")

Countries to process: 1
Relations to process filtered: 4145


### * make tests

In [12]:
config = {'root':ROOT, 's3':s3, 'logger':logger}

In [13]:
for country, df in dups_pending_process_df.items():
    logger.info(f"* Testing country {country}: {len(df)} relations")

    if not dups_test_state.get(country):
        dups_test_state[country] = {"to_process": set(), "processed": set(), "failed": set(), "next_index": 0}
    country_test_state = dups_test_state[country]

    total = len(df)
    test_res = {}
    
    CHUNK_SIZE = 15
    MAX_SECONDS_WITHOUT_UPLOAD = 300 # 5 minutes
    chunk_count = 0
    last_upload_time = time.time()
    save_path = TEST_DUPLICATES_DIR / country / f"{country}_first_level_test_res_{country_test_state['next_index']}.pkl"

    for i, (idx, row) in enumerate(df.iterrows(), start=1):
        id_triplet = (row["id"], row["tags.parent_id"], row["tags.country_id"])
        logger.info(f" ^ [{i}/{total}]: testing {id_triplet}:")

        # res = too.is_child_inside_parent(row["id"], row["tags.parent_id"])
        res = {'admin_centre': {'status': 'ok',
  'result': False,
  'status_type': None,
  'node': ['admin_centre', 256304928]},
 'label': {'status': 'ok',
  'result': False,
  'status_type': None,
  'node': ['label', 565543192]}}
        test_res[id_triplet] = res

        status_list = [v['status'] for k,v in res.items()]
        if 'error' in status_list:
            country_test_state['failed'].add(id_triplet)
        else:
            country_test_state['processed'].add(id_triplet)
            country_test_state['failed'].discard(id_triplet)

        time.sleep(2)
        resume  = {k:v['status'] for k,v in res.items()}
        logger.info(f" $ finished: status: {resume}")

        chunk_count += 1
        #* persist partial results
        if (chunk_count >= CHUNK_SIZE or (time.time() - last_upload_time) >= MAX_SECONDS_WITHOUT_UPLOAD):
            logger.info(f"* Checkpoint upload for {country}")
            # persist current chunk
            tgm.dump(save_path, test_res)
            # advance chunk
            test_res = {}
            country_test_state['next_index'] += 1
            # next chunk file
            save_path = TEST_DUPLICATES_DIR / country / f"{country}_first_level_test_res_{country_test_state['next_index']}.pkl"

            tgm.dump(DATA_DIR / "dups_test_state.json", dups_test_state)
            # upload to B2
            if not DEV_MODE:
                logger.info("* Uploading data to backblaze b2")
                tsm.upload_dir_files_to_backblaze(TEST_DUPLICATES_DIR / country, config)
                tsm.commit_file(DATA_DIR  / "dups_test_state.json", f"Update {country} first level test state: chunk {country_test_state['next_index'] - 1}", logger)

            chunk_count = 0
            last_upload_time = time.time()


    logger.info(f"* Finished {country}: saving data ...")
    # save test res
    if test_res:
        tgm.dump(save_path, test_res)
        country_test_state['next_index'] += 1

    # save country state
    tgm.dump(DATA_DIR / "dups_test_state.json", dups_test_state)

    if len(dups_test_state[country]['failed']) < 1:
        tsm.update_process_state(process_state, country, task, process_status='ok')
        tgm.dump(DATA_DIR / "process_state.json", process_state)

    # upload and commit after a country finishes
    if not DEV_MODE:
        logger.info("* Uploading data to backblaze b2")
        tsm.upload_dir_files_to_backblaze(TEST_DUPLICATES_DIR / country, config)
        tsm.commit_file(DATA_DIR  / "dups_test_state.json", f"Update {country} first level test state", logger)
        tsm.commit_file(DATA_DIR / "process_state.json", f"Update process state for {country}: ({task}, ok)", logger)

[INFO] : * Testing country UnitedStates: 4145 relations
[INFO] :  ^ [1/4145]: testing ('185175', '1822204', '148838'):
[INFO] :  $ finished: status: {'admin_centre': 'ok', 'label': 'ok'}
[INFO] :  ^ [2/4145]: testing ('112777', '1411346', '148838'):
[INFO] :  $ finished: status: {'admin_centre': 'ok', 'label': 'ok'}
[INFO] :  ^ [3/4145]: testing ('112875', '1411346', '148838'):
[INFO] :  $ finished: status: {'admin_centre': 'ok', 'label': 'ok'}
[INFO] :  ^ [4/4145]: testing ('112912', '1411346', '148838'):
[INFO] :  $ finished: status: {'admin_centre': 'ok', 'label': 'ok'}
[INFO] :  ^ [5/4145]: testing ('112928', '1411346', '148838'):
[INFO] :  $ finished: status: {'admin_centre': 'ok', 'label': 'ok'}
[INFO] :  ^ [6/4145]: testing ('112930', '1411346', '148838'):
[INFO] :  $ finished: status: {'admin_centre': 'ok', 'label': 'ok'}
[INFO] :  ^ [7/4145]: testing ('112932', '1411346', '148838'):
[INFO] :  $ finished: status: {'admin_centre': 'ok', 'label': 'ok'}
[INFO] :  ^ [8/4145]: testi

KeyboardInterrupt: 

In [ ]:
('5320537', '9583346', '286393')

### * [dev] review

In [ ]:
files = [f for f in (TEST_RESULTS_DIR / 'osm duplicates test').glob('*/*')]
print(len(files))
test_res_by_cntr = {}
for f in files:
    test_res_by_cntr[f.parent.name] = tgm.load(str(f))

print(len(test_res_by_cntr))

test_res_by_elem = {k:v for list in test_res_by_cntr.values() for k,v in list.items()}
print(len(test_res_by_elem))

30
30
1819


In [ ]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in test_res_by_elem.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                        1480
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, ok], [centroid, ok]]               154
[[admin_centre, missing], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                         151
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]                               26
[[admin_centre, missing], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]                      6
[[admin_centre, missing], [label, missing], [place, missing], [geom_node, missing], [centroid, missing]]       2
Name: count, dtype: int64

In [ ]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         6980
missing    2115
Name: count, dtype: int64

### * update processed data

In [ ]:
dups_test_processed = [ele for list in list(test_res_by_cntr.values()) for ele in list]
print(len(dups_test_processed))

1819


In [ ]:
tgm.dump((TEST_RESULTS_DIR / 'osm duplicates test' / "dups_test_processed.pkl"), dups_test_processed)